## Taxi Equilibrium Script

Using the parameters below, this script finds the equilibria licences and fare structure (price per km, per hour and fixed fee) of each city.

Parameters for each city:
- Rome: $\gamma=1.85$, $\tau=0.047$, $N=67000$, $\omega=40$, $\Theta=8$, $c_d=0.136$, $c_t=1.7$, $z=0.5$
- Milan: $\gamma=1.46$, $\tau=0.033$, $N=50000$, $\omega=45$, $\Theta=8$, $c_d=0.136$, $c_t=1.7$, $z=0.5$
- Naples: $\gamma=1.62$, $\tau=0.04$, $N=8667$, $\omega=30$, $\Theta=8$, $c_d=0.136$, $c_t=1.7$, $z=0.5$

In [5]:
from tabulate import tabulate

class City:
    def __init__(self, gamma,tau,N,omega,pi,Theta,cd,ct,z):
        self.gamma=gamma
        self.tau=tau
        self.N=N
        self.omega=omega
        self.pi=pi
        self.Theta=Theta
        self.cd=cd
        self.ct=ct
        self.z=z
    
    def licence(self):
        gamma,tau,N,omega,pi,Theta,cd,ct=self.gamma,self.tau,self.N,self.omega,self.pi,self.Theta,self.cd,self.ct
        J=N #initial guess
        for t in range(100):
            j=((omega*N)*tau*gamma**(2*(J/N)))/((gamma**(J/N)+1)*((pi+Theta*ct)*tau*gamma**(J/N)+Theta*(cd-ct*tau)))
            J=j
        return(int(round(J)))

    def d_cost(self):
        cd, ct, tau, gamma, J, N = self.cd, self.ct, self.tau, self.gamma, self.licence(), self.N
        M=cd+ct*tau*(gamma**(J/N)-1)
        return(M)

    def p_t(self):
        pi, Theta, tau, M, gamma, J, N = self.pi, self.Theta, self.tau, self.d_cost(), self.gamma, self.licence(), self.N
        pt=pi/Theta+M/tau*gamma**(J/N)
        return(round(pt,2))

    def p_d(self):
        pt, tau= self.p_t(), self.tau
        pd=pt*tau
        return(round(pd,2))

    def fix(self):
        pt, tau, gamma, J, N, z = self.p_t(), self.tau, self.gamma, self.licence(), self.N, self.z
        k=z*pt*tau*gamma**(J/N)
        return(round(k,2))

    def short(self):
        s=(self.p_t()*(5/60)+self.p_d()*5+self.fix())
        return(round(s,2))

    def long(self):
        l=(self.p_t()*(1/6)+self.p_d()*10+self.fix())
        return(round(l,2))                  

Rome=City(gamma=1.85, tau=0.047, N=67000, omega=40, pi=104, Theta=8, cd=0.136, ct=1.7, z=0.5)
Milan=City(gamma=1.46, tau=0.033, N=50000, omega=45, pi=125, Theta=8, cd=0.136, ct=1.7, z=0.5)
Naples=City(gamma=1.62, tau=0.04, N=8667, omega=30, pi=83, Theta=8, cd=0.136, ct=1.7, z=0.5)

e = tabulate([
    ['Licence',Rome.licence(), Milan.licence(), Naples.licence()],
    ['Hr price',Rome.p_t(), Milan.p_t(), Naples.p_t()],
    ['Km price',Rome.p_d(), Milan.p_d(), Naples.p_d()],
    ['Fixed fare',Rome.fix(), Milan.fix(), Naples.fix()],
    ['Short trip',Rome.short(), Milan.short(), Naples.short()],
    ['Long trip',Rome.long(), Milan.long(), Naples.long()]
], headers=['Rome', 'Milan', 'Naples'],
colalign=("left", "left", "left", "left"))

print(e)

            Rome    Milan    Naples
----------  ------  -------  --------
Licence     11160   7369     1230
Hr price    16.41   20.09    14.14
Km price    0.77    0.66     0.57
Fixed fare  0.43    0.35     0.3
Short trip  5.65    5.32     4.33
Long trip   10.87   10.3     8.36
